# Module 5 — Modules, Files and Error Handling

**Python for Machine Learning · MSc Artificial Intelligence · Swansea University**

---

## Learning objectives

1. Import modules and packages; know the most useful parts of the **standard library**.
2. Install packages with `pip` and isolate projects with **virtual environments**.
3. Read and write **text, CSV, JSON and pickle** files, and work with paths using `pathlib`.
4. Handle errors with `try` / `except` / `else` / `finally`, and raise your own exceptions.
5. Create your own reusable module.

---

## 5.1 Modules and imports

A **module** is a `.py` file containing definitions. A **package** is a folder of modules. Python ships with a large **standard library**, and thousands of third-party packages (NumPy, pandas, scikit-learn…) are installed with `pip`.

In [1]:
import math                          # import the whole module
print(math.sqrt(16), math.pi, math.log(math.e))

from math import exp, log            # import specific names
print(exp(1), log(100, 10))

import statistics as stats           # import with an alias
print(stats.median([3, 1, 4, 1, 5]))

4.0 3.141592653589793 1.0
2.718281828459045 2.0
3


The ML community uses standard aliases — **always use these**, everyone will recognise them:

```text
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
```

> Avoid `from module import *`: it dumps many unknown names into your namespace and can silently overwrite your own variables.

## 5.2 A tour of the standard library

### `random` — pseudo-random numbers and reproducibility

In [2]:
import random

random.seed(42)                       # fix the seed -> same "random" numbers every run
print(random.random())                # float in [0, 1)
print(random.randint(1, 6))           # integer in [1, 6], both ends included
print(random.choice(["cat", "dog", "bird"]))
print(random.sample(range(100), 5))   # 5 distinct values

data = list(range(10))
random.shuffle(data)                  # shuffles IN PLACE
print(data)
print(random.gauss(0, 1))             # normal distribution, mean 0, std 1

0.6394267984578837
1
bird
[35, 31, 28, 17, 94]
[7, 2, 5, 6, 0, 3, 4, 9, 8, 1]
0.14759598253598957


**Reproducibility** — setting seeds so experiments can be repeated exactly — is essential in ML research. You will set seeds in `random`, NumPy and scikit-learn (`random_state=`).

### `collections` and `itertools`

In [3]:
from collections import Counter, defaultdict, namedtuple, deque
import itertools

Point = namedtuple("Point", ["x", "y"])
p = Point(3, 4)
print(p.x, p.y, p)

recent = deque(maxlen=3)              # keeps only the last 3 items (e.g. a moving window)
for loss in [0.9, 0.8, 0.7, 0.6]:
    recent.append(loss)
print(recent)

grid = {"lr": [0.1, 0.01], "depth": [3, 5]}
for combo in itertools.product(*grid.values()):     # every combination: grid search!
    print(dict(zip(grid.keys(), combo)))

print(list(itertools.combinations(["a", "b", "c"], 2)))
print(list(itertools.accumulate([1, 2, 3, 4])))      # running totals

3 4 Point(x=3, y=4)
deque([0.8, 0.7, 0.6], maxlen=3)
{'lr': 0.1, 'depth': 3}
{'lr': 0.1, 'depth': 5}
{'lr': 0.01, 'depth': 3}
{'lr': 0.01, 'depth': 5}
[('a', 'b'), ('a', 'c'), ('b', 'c')]
[1, 3, 6, 10]


### `time` and `datetime`

In [4]:
import time
from datetime import datetime, timedelta

start = time.perf_counter()
_ = sum(i * i for i in range(500_000))
print(f"elapsed: {time.perf_counter() - start:.4f} s")

now = datetime.now()
print(now.strftime("%Y-%m-%d %H:%M"))           # format a date as text
print((now + timedelta(days=7)).date())
run_name = f"experiment_{now:%Y%m%d_%H%M%S}"     # timestamped names for experiment logs
print(run_name)

elapsed: 0.0523 s
2026-10-31 17:36
2026-11-07
experiment_20261031_173600


## 5.3 Installing packages and virtual environments

These commands are run in a **terminal**, not in Python (in Jupyter you can prefix them with `!` or `%`):

```text
pip install numpy pandas matplotlib scikit-learn     # install
pip install scikit-learn==1.5.0                      # a specific version
pip list                                             # what is installed
pip freeze > requirements.txt                        # save exact versions
pip install -r requirements.txt                      # recreate them elsewhere
```

A **virtual environment** is an isolated Python installation per project, so that projects needing different library versions do not conflict:

```text
python -m venv .venv               # create
source .venv/bin/activate          # activate (macOS / Linux)
.venv\Scripts\activate             # activate (Windows)
deactivate                         # leave
```

Conda (`conda create -n ml python=3.12`) is a popular alternative, especially for GPU libraries.

## 5.4 Paths with `pathlib`

`pathlib` handles file paths in an object-oriented and operating-system-independent way (no more worrying about `/` versus `\`).

In [5]:
from pathlib import Path

data_dir = Path("course_data")
data_dir.mkdir(exist_ok=True)              # create the folder if it doesn't exist
file_path = data_dir / "notes.txt"         # the '/' operator joins paths
print(file_path, file_path.suffix, file_path.stem, file_path.parent)
print(Path.cwd())                          # current working directory

course_data\notes.txt .txt notes course_data
C:\Users\nikol\Downloads\jupyter


## 5.5 Text files

Always open files with a `with` statement (a **context manager**). It closes the file automatically — even if an error occurs.

| Mode | Meaning |
|---|---|
| `"r"` | read (default) — error if the file is missing |
| `"w"` | write — **overwrites** existing content |
| `"a"` | append to the end |
| `"rb"` / `"wb"` | read / write binary (e.g. pickle) |

In [6]:
with open(file_path, "w", encoding="utf-8") as f:
    f.write("epoch,loss\n")
    for epoch, loss in enumerate([0.9, 0.7, 0.5]):
        f.write(f"{epoch},{loss}\n")

with open(file_path, "a", encoding="utf-8") as f:
    f.write("3,0.4\n")

with open(file_path, encoding="utf-8") as f:
    content = f.read()             # the whole file as one string
print(content)

epoch,loss
0,0.9
1,0.7
2,0.5
3,0.4



Reading line by line is memory-efficient for large files — a file object is iterable:

In [7]:
with open(file_path, encoding="utf-8") as f:
    header = f.readline().strip()
    for line in f:                 # continues after the header
        epoch, loss = line.strip().split(",")
        print(int(epoch), float(loss))

0 0.9
1 0.7
2 0.5
3 0.4


`pathlib` also offers shortcuts for small files:

In [8]:
(data_dir / "hello.txt").write_text("Hello from pathlib\n", encoding="utf-8")
print((data_dir / "hello.txt").read_text(encoding="utf-8"))
print(sorted(p.name for p in data_dir.glob("*.txt")))   # list matching files

Hello from pathlib

['hello.txt', 'notes.txt']


## 5.6 CSV files

CSV (comma-separated values) is the most common format for tabular data. In practice you will use pandas (`pd.read_csv`), but the built-in `csv` module correctly handles quotes and commas inside fields:

In [9]:
import csv

rows = [
    {"name": "Ana", "grade": 72, "comment": "excellent, very consistent"},
    {"name": "Ben", "grade": 65, "comment": "good"},
]
csv_path = data_dir / "grades.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "grade", "comment"])
    writer.writeheader()
    writer.writerows(rows)

print(csv_path.read_text(encoding="utf-8"))

with open(csv_path, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(row["name"], int(row["grade"]))   # NOTE: everything is read as str

name,grade,comment
Ana,72,"excellent, very consistent"
Ben,65,good

Ana 72
Ben 65


## 5.7 JSON files

JSON is the standard format for configurations, API responses and experiment results. It maps directly onto Python dicts, lists, strings, numbers, `True/False` and `None`.

In [10]:
import json

experiment = {
    "model": "RandomForest",
    "params": {"n_estimators": 200, "max_depth": None},
    "scores": [0.91, 0.93, 0.92],
    "converged": True,
}

json_path = data_dir / "experiment.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(experiment, f, indent=2)

with open(json_path, encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded["params"]["n_estimators"])
print(json.dumps(loaded, indent=2))    # dumps/loads work with strings instead of files

200
{
  "model": "RandomForest",
  "params": {
    "n_estimators": 200,
    "max_depth": null
  },
  "scores": [
    0.91,
    0.93,
    0.92
  ],
  "converged": true
}


## 5.8 Pickle — saving arbitrary Python objects

`pickle` serialises (almost) any Python object to bytes — for example a trained model. `joblib` (used by scikit-learn) is similar and better for large NumPy arrays.

In [11]:
import pickle

model_state = {"weights": [0.2, -1.3, 0.7], "bias": 0.05}
with open(data_dir / "model.pkl", "wb") as f:
    pickle.dump(model_state, f)

with open(data_dir / "model.pkl", "rb") as f:
    restored = pickle.load(f)
print(restored)

{'weights': [0.2, -1.3, 0.7], 'bias': 0.05}


> ⚠️ **Security:** never unpickle files from untrusted sources — loading a pickle can execute arbitrary code.

## 5.9 Errors and exceptions

There are two kinds of errors:

* **Syntax errors** — the code is not valid Python; nothing runs.
* **Exceptions** — the code is valid, but something goes wrong *while running*.

| Exception | Typical cause |
|---|---|
| `NameError` | using a variable that doesn't exist (typo, cell not run) |
| `TypeError` | wrong type, e.g. `"a" + 1` |
| `ValueError` | right type, bad value, e.g. `int("abc")` |
| `IndexError` | list index out of range |
| `KeyError` | missing dictionary key |
| `ZeroDivisionError` | division by zero |
| `FileNotFoundError` | file path is wrong |
| `AttributeError` | object has no such attribute/method |

### Reading a traceback

When an exception is not handled, Python prints a **traceback**. Read it **from the bottom up**: the last line gives the exception type and message; the lines above show where it happened. For example:

```text
Traceback (most recent call last):
  Cell In[3], line 4
    print(average([]))
  Cell In[3], line 2, in average
    return sum(xs) / len(xs)
ZeroDivisionError: division by zero
```

### `try` / `except`

In [12]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Warning: division by zero, returning nan")
        return float("nan")

print(safe_divide(10, 2))
print(safe_divide(1, 0))

5.0
nan


Catch **specific** exceptions. A bare `except:` hides real bugs (even typos!).

In [13]:
raw_values = ["3.2", "4.1", "n/a", "5.0", "", "2.8"]
clean = []
for v in raw_values:
    try:
        clean.append(float(v))
    except ValueError:
        print(f"skipping invalid value {v!r}")   # !r shows the repr (with quotes)
print(clean)

skipping invalid value 'n/a'
skipping invalid value ''
[3.2, 4.1, 5.0, 2.8]


### The full form: `else` and `finally`

In [14]:
def read_number(path):
    try:
        f = open(path, encoding="utf-8")
    except FileNotFoundError as e:       # 'as e' gives access to the exception object
        print("Could not open file:", e)
        return None
    else:                                # runs only if NO exception happened
        with f:
            return float(f.read())
    finally:                             # ALWAYS runs - for clean-up
        print(f"(finished trying {path})")

(data_dir / "number.txt").write_text("3.14")
print(read_number(data_dir / "number.txt"))
print(read_number(data_dir / "missing.txt"))

(finished trying course_data\number.txt)
3.14
Could not open file: [Errno 2] No such file or directory: 'course_data\\missing.txt'
(finished trying course_data\missing.txt)
None


Several exception types can be handled together, or separately:

In [15]:
for value in ["10", "zero", "0"]:
    try:
        result = 100 / int(value)
    except (ValueError, ZeroDivisionError) as e:
        print(f"{value!r}: {type(e).__name__}: {e}")
    else:
        print(f"{value!r}: {result}")

'10': 10.0
'zero': ValueError: invalid literal for int() with base 10: 'zero'
'0': ZeroDivisionError: division by zero


### Raising exceptions

Your own functions should **fail loudly** on invalid input, with a clear message. This prevents silent bugs, e.g. training on data with the wrong shape.

In [16]:
def train_test_split_simple(data, test_size=0.2):
    if not 0 < test_size < 1:
        raise ValueError(f"test_size must be between 0 and 1, got {test_size}")
    if len(data) < 2:
        raise ValueError("need at least 2 samples")
    n_test = max(1, int(len(data) * test_size))
    return data[:-n_test], data[-n_test:]

print(train_test_split_simple(list(range(10))))
try:
    train_test_split_simple(list(range(10)), test_size=1.5)
except ValueError as e:
    print("ValueError:", e)

([0, 1, 2, 3, 4, 5, 6, 7], [8, 9])
ValueError: test_size must be between 0 and 1, got 1.5


### Custom exceptions

In [17]:
class NotFittedError(Exception):
    """Raised when a model is used before being trained."""

def predict(model, x):
    if not model.get("fitted", False):
        raise NotFittedError("call fit() before predict()")
    return model["w"] * x

try:
    predict({"fitted": False}, 3)
except NotFittedError as e:
    print("NotFittedError:", e)

NotFittedError: call fit() before predict()


(scikit-learn has exactly such an exception: `sklearn.exceptions.NotFittedError`.)

### `assert` — sanity checks while developing

In [18]:
X = [[1, 2], [3, 4], [5, 6]]
y = [0, 1, 0]
assert len(X) == len(y), "X and y must have the same number of samples"
print("shapes OK")

shapes OK


`assert` is for catching *programming* mistakes during development. For validating user input, use `raise` (asserts can be disabled with `python -O`).

### Warnings

Libraries often issue **warnings** (e.g. a model did not converge). They do not stop execution:

In [19]:
import warnings

def fit(max_iter):
    if max_iter < 100:
        warnings.warn("max_iter is small; the model may not converge", UserWarning)
    return "fitted"

print(fit(10))

fitted


C:\Users\nikol\AppData\Local\Temp\ipykernel_5612\3844846997.py:5: UserWarning: max_iter is small; the model may not converge
  warnings.warn("max_iter is small; the model may not converge", UserWarning)


## 5.10 Writing your own module

Any `.py` file can be imported. Below we create one from Python code (normally you would write it in an editor).

In [20]:
module_code = '''"""Small metrics module for the course."""

def accuracy(y_true, y_pred):
    """Fraction of matching labels."""
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)

def mse(y_true, y_pred):
    """Mean squared error."""
    return sum((t - p) ** 2 for t, p in zip(y_true, y_pred)) / len(y_true)

if __name__ == "__main__":
    # Runs only when executed as a script (python my_metrics.py), NOT when imported.
    print("self-test:", accuracy([1, 0], [1, 1]))
'''
Path("my_metrics.py").write_text(module_code, encoding="utf-8")

import my_metrics
print(my_metrics.accuracy([1, 0, 1], [1, 1, 1]))
print(my_metrics.mse([1.0, 2.0], [1.5, 2.5]))
help(my_metrics)

0.6666666666666666
0.25
Help on module my_metrics:

NAME
    my_metrics - Small metrics module for the course.

FUNCTIONS
    accuracy(y_true, y_pred)
        Fraction of matching labels.

    mse(y_true, y_pred)
        Mean squared error.

FILE
    c:\users\nikol\downloads\jupyter\my_metrics.py




The `if __name__ == "__main__":` idiom lets a file act both as an importable module and as a runnable script.

> If you edit a module after importing it, Jupyter keeps the old version. Restart the kernel, or use `%load_ext autoreload` and `%autoreload 2`.

A typical ML project structure:

```text
my_project/
├── data/               raw and processed data
├── notebooks/          exploration
├── src/
│   ├── __init__.py     marks the folder as a package
│   ├── data.py         loading & cleaning
│   ├── features.py
│   └── models.py
├── requirements.txt
└── README.md
```

---

## Summary

* `import module`, `from module import name`, `import module as alias`.
* The standard library includes `math`, `random`, `collections`, `itertools`, `pathlib`, `csv`, `json`, `pickle`…
* Open files with `with open(...) as f:`; use `pathlib.Path` for paths.
* Handle **specific** exceptions with `try/except`; use `else` for success and `finally` for clean-up.
* `raise` meaningful exceptions for invalid input.

---

## Exercises

Solutions: `solutions/05_modules_files_errors_solutions.ipynb`.

**Exercise 5.1 — Reproducible randomness.**
(a) Set the seed to `0` and generate a list of 5 random integers between 1 and 100. (b) Do it again with the same seed and verify (with `==`) that you get the same list. (c) Write `random_split(data, test_size, seed)` that shuffles a **copy** of `data` with the given seed and returns `(train, test)`.

In [21]:
import random
# Your code here

**Exercise 5.2 — Write and read a log file.**
(a) Write the losses below to `course_data/training_log.txt`, one line per epoch in the format `epoch=1 loss=2.3000`. (b) Read the file back, parse each line, and build a list of floats. (c) Print the epoch with the lowest loss.

In [22]:
losses = [2.3, 1.7, 1.2, 0.95, 0.97, 0.88]
# Your code here

**Exercise 5.3 — JSON config.**
(a) Save the `config` dictionary to `course_data/config.json` (indented). (b) Load it back, change `epochs` to 50, add a key `"optimizer": "adam"` and save it again. (c) Load once more and print it.

In [23]:
config = {"model": "MLP", "hidden_layers": [64, 32], "epochs": 20, "lr": 0.001}
# Your code here

**Exercise 5.4 — Robust parsing.**
Write `parse_measurements(strings)` which returns a list of floats, **skipping** invalid entries but **counting** them. Return a tuple `(values, n_invalid)`. Test with the list below.

In [24]:
raw = ["1.5", "2.0", "abc", "", "3.25", "NaN?", "4", None]
# Your code here (hint: float(None) raises a TypeError, not a ValueError)

**Exercise 5.5 — Input validation.**
Write `standardise(values)` that returns $(x - \mu)/\sigma$ for each value (population std). It should raise a `TypeError` if `values` is not a list or tuple, a `ValueError` if it has fewer than 2 elements, and a `ValueError` if the standard deviation is 0. Test each failure case with `try/except`.

In [25]:
# Your code here

**Exercise 5.6 — CSV round-trip.**
Write the `results` below to `course_data/results.csv` with the `csv` module (header: `model,accuracy,f1`). Read it back with `csv.DictReader`, convert the numbers to `float`, and print the model with the highest F1 score.

In [26]:
results = [("LogReg", 0.88, 0.86), ("SVM", 0.91, 0.90), ("Tree", 0.84, 0.85)]
# Your code here

**Exercise 5.7 — Grid search with itertools (challenge).**
Using `itertools.product`, build the list of all hyper-parameter combinations of `grid` as dictionaries. Print how many there are and the first three.

In [27]:
grid = {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"], "gamma": ["scale", "auto"]}
# Your code here